## 1. ScanNet Dataset Setup

Downloads and extracts ScanNet's preprocessed `scannet_frames_25k` archive (25,000 RGB-D frame pairs across all scenes), strips out everything except the `color`/`depth` folders per scene to save space, then builds a shuffled, seeded (`seed=42`) split into four fixed slices — `batch1`/`batch2`/`batch3` (9,000 frames each, available for training/calibration use) and `test_samples` (the remaining ~1,000+ frames, used as the held-out evaluation set for every model comparison below).

**Note on `test_samples` and the ScanNet-stage checkpoint:** since `20_epochs_scannet_documentation_model.pth` was trained on the full ScanNet dataset by design (a deployment-oriented checkpoint, not a benchmarking one), frames in `test_samples` may not be strictly held out from what that specific checkpoint saw during training — keep that in mind when reading its results here versus the Base/NYUv2-stage checkpoints, which were not trained on ScanNet at all.

In [ ]:
!wget http://kaldir.vc.cit.tum.de/scannet/download-scannet.py -O download-scannet.py

--2026-09-04 09:13:52--  http://kaldir.vc.cit.tum.de/scannet/download-scannet.py
Resolving kaldir.vc.cit.tum.de (kaldir.vc.cit.tum.de)... 131.159.98.129, 2a09:80c0:98::129
Connecting to kaldir.vc.cit.tum.de (kaldir.vc.cit.tum.de)|131.159.98.129|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://kaldir.vc.cit.tum.de:443/scannet/download-scannet.py [following]
--2026-09-04 09:13:53--  https://kaldir.vc.cit.tum.de/scannet/download-scannet.py
Connecting to kaldir.vc.cit.tum.de (kaldir.vc.cit.tum.de)|131.159.98.129|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13409 (13K) [text/x-python]
Saving to: ‘download-scannet.py’

download-scannet.py 100%[===================>]  13.09K  --.-KB/s    in 0s      

2026-09-04 09:13:54 (182 MB/s) - ‘download-scannet.py’ saved [13409/13409]



In [ ]:
import os
os.makedirs("/content/tasks", exist_ok=True)

In [ ]:
!wget -c "http://kaldir.vc.cit.tum.de/scannet//v2/tasks/scannet_frames_25k.zip" -O /content/tasks/scannet_frames_25k.zip

--2026-09-04 09:13:54--  http://kaldir.vc.cit.tum.de/scannet//v2/tasks/scannet_frames_25k.zip
Resolving kaldir.vc.cit.tum.de (kaldir.vc.cit.tum.de)... 131.159.98.129, 2a09:80c0:98::129
Connecting to kaldir.vc.cit.tum.de (kaldir.vc.cit.tum.de)|131.159.98.129|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://kaldir.vc.cit.tum.de:443/scannet/v2/tasks/scannet_frames_25k.zip [following]
--2026-09-04 09:13:54--  https://kaldir.vc.cit.tum.de/scannet/v2/tasks/scannet_frames_25k.zip
Connecting to kaldir.vc.cit.tum.de (kaldir.vc.cit.tum.de)|131.159.98.129|:443... connected.
HTTP request sent, awaiting response... 206 Partial Content
Length: 6026705560 (5.6G), 1067727512 (1018M) remaining [application/zip]
Saving to: ‘/content/tasks/scannet_frames_25k.zip’

/content/tasks/scan 100%[++++++++++++++++===>]   5.61G  21.4MB/s    in 49s     

2026-09-04 09:14:44 (20.7 MB/s) - ‘/content/tasks/scannet_frames_25k.zip’ saved [6026705560/6026705560]



In [ ]:
import os
print(os.listdir("/content/tasks"))

['scannet_frames_25k.zip']


In [ ]:
import zipfile
import os

zip_path = "/content/tasks/scannet_frames_25k.zip"
extract_path = "/content/scannet_frames_25k"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed.")

Extraction completed.


In [ ]:
root_path= "/content/scannet_frames_25k/scannet_frames_25k"

In [ ]:
from re import sub
import os
folders = []
for folder in os.listdir(root_path):
  folder_path = os.path.join(root_path, folder)
  if os.path.isdir(folder_path):
    folders.append(folder)
print(folders)

['scene0666_00', 'scene0100_02', 'scene0093_01', 'scene0435_01', 'scene0069_00', 'scene0547_01', 'scene0600_01', 'scene0685_02', 'scene0237_01', 'scene0254_00', 'scene0256_02', 'scene0413_00', 'scene0174_00', 'scene0337_02', 'scene0160_04', 'scene0209_02', 'scene0448_00', 'scene0601_01', 'scene0170_02', 'scene0697_01', 'scene0662_02', 'scene0120_01', 'scene0502_00', 'scene0498_01', 'scene0470_01', 'scene0299_00', 'scene0340_01', 'scene0115_00', 'scene0556_01', 'scene0512_00', 'scene0630_06', 'scene0416_03', 'scene0274_02', 'scene0542_00', 'scene0250_00', 'scene0286_01', 'scene0456_01', 'scene0666_01', 'scene0690_01', 'scene0004_00', 'scene0385_02', 'scene0643_00', 'scene0266_01', 'scene0303_00', 'scene0195_01', 'scene0362_03', 'scene0134_00', 'scene0424_01', 'scene0508_02', 'scene0572_00', 'scene0430_01', 'scene0161_01', 'scene0394_00', 'scene0136_02', 'scene0664_00', 'scene0143_02', 'scene0246_00', 'scene0376_01', 'scene0312_00', 'scene0207_02', 'scene0351_00', 'scene0362_01', 'scene0

In [ ]:
import os
import shutil

keep_folders = {"color", "depth"}

for scene in sorted(os.listdir(root_path)):
    scene_path = os.path.join(root_path, scene)
    if not os.path.isdir(scene_path):
        continue

    for item in os.listdir(scene_path):
        item_path = os.path.join(scene_path, item)

        if item in keep_folders:
            continue  # keep these

        if os.path.isdir(item_path):
            shutil.rmtree(item_path)
        else:
            os.remove(item_path)

print("Cleanup complete.")

Cleanup complete.


In [ ]:
import os
samples = []
for folder in sorted(folders):
    folder_path = os.path.join(root_path, folder)
    color_dir = os.path.join(folder_path, "color")
    depth_dir = os.path.join(folder_path, "depth")
    if not os.path.isdir(color_dir) or not os.path.isdir(depth_dir):
        continue
    color_lookup = {}
    for color_file in sorted(os.listdir(color_dir)):
        color_name = os.path.splitext(color_file)[0]
        color_path = os.path.join(color_dir, color_file)
        color_lookup[color_name] = color_path
    for depth_file in sorted(os.listdir(depth_dir)):
        depth_name = os.path.splitext(depth_file)[0]
        if depth_name in color_lookup:
            depth_path = os.path.join(depth_dir, depth_file)
            samples.append(
                (
                    color_lookup[depth_name],
                    depth_path
                )
            )
print(f"Total RGB-Depth pairs: {len(samples)}")
print(samples[:5])

Total RGB-Depth pairs: 24902
[('/content/scannet_frames_25k/scannet_frames_25k/scene0000_00/color/000000.jpg', '/content/scannet_frames_25k/scannet_frames_25k/scene0000_00/depth/000000.png'), ('/content/scannet_frames_25k/scannet_frames_25k/scene0000_00/color/000100.jpg', '/content/scannet_frames_25k/scannet_frames_25k/scene0000_00/depth/000100.png'), ('/content/scannet_frames_25k/scannet_frames_25k/scene0000_00/color/000200.jpg', '/content/scannet_frames_25k/scannet_frames_25k/scene0000_00/depth/000200.png'), ('/content/scannet_frames_25k/scannet_frames_25k/scene0000_00/color/000300.jpg', '/content/scannet_frames_25k/scannet_frames_25k/scene0000_00/depth/000300.png'), ('/content/scannet_frames_25k/scannet_frames_25k/scene0000_00/color/000400.jpg', '/content/scannet_frames_25k/scannet_frames_25k/scene0000_00/depth/000400.png')]


In [ ]:
import random
random.seed(42)
shuffled = list(samples)
random.shuffle(shuffled)

batch1_samples = shuffled[:9000]
batch2_samples = shuffled[9000:18000]
batch3_samples = shuffled[18000:24000]
test_samples   = shuffled[24000:]

## 2. YOLO26-Depth — Offline Evaluation (PyTorch)

Runs a fine-tuned YOLO26-Depth checkpoint (downloaded from Hugging Face) directly through Ultralytics' own `YOLO()` inference wrapper, over every frame in `test_samples`. Metrics: AbsRel, RMSE, LogRMSE, Delta1-3, same math as every other evaluation in this notebook, so results are directly comparable across models and precisions.

In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 6.6 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import hf_hub_download

checkpoint_path = hf_hub_download(
    repo_id="WasiqSaleem/Fine-Tuned-Depth-Estimation-for-ROS-2",
    filename="nano_yolo26-depth-scannet-batch3.pt",
    repo_type="model",
    local_dir="/content"
)

print(f"Checkpoint downloaded to: {checkpoint_path}")

nano_yolo26-depth-scannet-batch3.pt: reconstructing file:   0%|          |  0.00B / 10.6MB            

nano_yolo26-depth-scannet-batch3.pt: downloading bytes:           |  0.00B            

Checkpoint downloaded to: /content/nano_yolo26-depth-scannet-batch3.pt


In [ ]:
import os
import csv
import numpy as np
from PIL import Image as PILImage
from ultralytics import YOLO

def run_offline_eval(model_path, test_samples, output_csv, depth_scale=1.0 / 1000.0, min_depth=0.1, max_depth=20.0, imgsz=640):
    print(f"Loading model: {model_path}")
    model = YOLO(model_path)

    output_dir = os.path.dirname(output_csv)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    fieldnames = ["frame_id", "color_path", "gt_mean", "pred_mean", "diff_mean", "AbsRel", "RMSE", "LogRMSE", "Delta1", "Delta2", "Delta3", "valid_pixel_frac"]
    rows = []
    n_skipped = 0

    for frame_id, (color_path, depth_path) in enumerate(test_samples):
        try:
            gt_raw = PILImage.open(depth_path)
            gt_array = np.array(gt_raw)
            gt = gt_array.astype(np.float32)

            if gt_array.dtype == np.uint16:
                gt *= depth_scale

            results = model(color_path, imgsz=imgsz, verbose=False)
            result = results[0]

            if result.depth is None:
                print(f"[{frame_id}] No depth output — skipping.")
                n_skipped += 1
                continue

            pred = np.squeeze(result.depth.data.cpu().numpy()).astype(np.float32)

            if pred.ndim != 2:
                print(f"[{frame_id}] Unexpected pred shape {pred.shape} — skipping.")
                n_skipped += 1
                continue

            if pred.shape != gt.shape:
                pred = np.array(PILImage.fromarray(pred).resize((gt.shape[1], gt.shape[0]), resample=PILImage.BILINEAR))

        except Exception as e:
            print(f"[{frame_id}] Failed to load/infer: {e}")
            n_skipped += 1
            continue

        mask = (gt >= min_depth) & (gt <= max_depth) & (pred >= min_depth) & (pred <= max_depth)

        if np.count_nonzero(mask) == 0:
            print(f"[{frame_id}] No valid pixels — skipping.")
            n_skipped += 1
            continue

        gt_valid = gt[mask]
        pred_valid = pred[mask]
        diff = np.abs(gt_valid - pred_valid)

        abs_rel = np.mean(diff / gt_valid)
        rmse = np.sqrt(np.mean((gt_valid - pred_valid) ** 2))
        gt_log = np.log(np.clip(gt_valid, 1e-6, None))
        pred_log = np.log(np.clip(pred_valid, 1e-6, None))
        log_rmse = np.sqrt(np.mean((gt_log - pred_log) ** 2))

        max_ratio = np.maximum(gt_valid / pred_valid, pred_valid / gt_valid)
        delta1 = np.mean(max_ratio < 1.25)
        delta2 = np.mean(max_ratio < 1.25 ** 2)
        delta3 = np.mean(max_ratio < 1.25 ** 3)

        rows.append({
            "frame_id": frame_id,
            "color_path": color_path,
            "gt_mean": float(np.mean(gt_valid)),
            "pred_mean": float(np.mean(pred_valid)),
            "diff_mean": float(np.mean(diff)),
            "AbsRel": float(abs_rel),
            "RMSE": float(rmse),
            "LogRMSE": float(log_rmse),
            "Delta1": float(delta1),
            "Delta2": float(delta2),
            "Delta3": float(delta3),
            "valid_pixel_frac": float(np.count_nonzero(mask) / mask.size),
        })

        if frame_id % 50 == 0:
            print(f"Frame {frame_id}/{len(test_samples)} | AbsRel: {abs_rel:.4f} | RMSE: {rmse:.4f} | D1: {delta1:.4f}")

    with open(output_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nWrote {len(rows)} rows to {output_csv} ({n_skipped} skipped)")

    if not rows:
        print("No frames evaluated — nothing to summarize.")
        return None

    keys = ["AbsRel", "RMSE", "LogRMSE", "Delta1", "Delta2", "Delta3"]
    means = {k: float(np.mean([r[k] for r in rows])) for k in keys}

    print(f"\n=== Aggregate over {len(rows)} frames ===")
    for k, v in means.items():
        print(f"{k}: {v:.4f}")

    return means

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [ ]:
means = run_offline_eval(
    model_path="/content/nano_yolo26-depth-scannet-batch3.pt",
    test_samples=test_samples,
    output_csv="/content/nano_scannet_test_eval.csv",
    depth_scale=1.0 / 1000.0,
    min_depth=0.1,
    max_depth=20.0,
    imgsz=640,
)

Loading model: /content/nano_yolo26-depth-scannet-batch3.pt
Frame 0/902 | AbsRel: 0.1041 | RMSE: 0.1936 | D1: 0.8914
Frame 50/902 | AbsRel: 0.2174 | RMSE: 0.2872 | D1: 0.6296
Frame 100/902 | AbsRel: 0.1566 | RMSE: 0.2837 | D1: 0.9102
Frame 150/902 | AbsRel: 0.1760 | RMSE: 0.4128 | D1: 0.7265
Frame 200/902 | AbsRel: 0.1397 | RMSE: 0.4301 | D1: 0.7710
Frame 250/902 | AbsRel: 0.1179 | RMSE: 0.3544 | D1: 0.8292
Frame 300/902 | AbsRel: 0.0660 | RMSE: 0.1553 | D1: 0.9849
Frame 350/902 | AbsRel: 0.1847 | RMSE: 0.4993 | D1: 0.7080
Frame 400/902 | AbsRel: 0.1086 | RMSE: 0.2952 | D1: 0.9348
Frame 450/902 | AbsRel: 0.2407 | RMSE: 0.4792 | D1: 0.4662
Frame 500/902 | AbsRel: 0.0845 | RMSE: 0.2376 | D1: 0.9947
Frame 550/902 | AbsRel: 0.1372 | RMSE: 0.1987 | D1: 0.9643
Frame 600/902 | AbsRel: 0.1336 | RMSE: 0.2603 | D1: 0.7742
Frame 650/902 | AbsRel: 0.0736 | RMSE: 0.1738 | D1: 0.9265
Frame 700/902 | AbsRel: 0.1683 | RMSE: 1.0787 | D1: 0.7983
Frame 750/902 | AbsRel: 0.0628 | RMSE: 0.1848 | D1: 0.9902

## 3. Depth Anything V2 — Offline Evaluation (PyTorch)

Same idea as the YOLO26 section above, but for DAv2: loads the model architecture directly (`DepthAnythingV2` class + `load_state_dict`) rather than through a wrapper library, since DAv2 has no Ultralytics-style inference API. `run_offline_eval_dav2` (defined in the next cell) mirrors `run_offline_eval`'s metric math exactly.

In [ ]:
!git clone https://github.com/DepthAnything/Depth-Anything-V2
%cd Depth-Anything-V2/metric_depth
!pip install -r requirements.txt

Cloning into 'Depth-Anything-V2'...
remote: Enumerating objects: 146, done.
remote: Total 146 (delta 0), reused 0 (delta 0), pack-reused 146 (from 1)
Receiving objects: 100% (146/146), 45.18 MiB | 13.07 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/Depth-Anything-V2/metric_depth
ERROR: Could not find a version that satisfies the requirement open3d (from versions: none)
ERROR: No matching distribution found for open3d


In [ ]:
import cv2
import torch

from depth_anything_v2.dpt import DepthAnythingV2

model_configs = {
    "vits": {"encoder": "vits", "features": 64,  "out_channels": [48, 96, 192, 384]},
    "vitb": {"encoder": "vitb", "features": 128, "out_channels": [96, 192, 384, 768]},
    "vitl": {"encoder": "vitl", "features": 256, "out_channels": [256, 512, 1024, 1024]},
}

encoder, max_depth = "vits", 20
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from huggingface_hub import hf_hub_download

checkpoint_path = hf_hub_download(
    repo_id="WasiqSaleem/Fine-Tuned-Depth-Estimation-for-ROS-2",
    filename="early_stopping_56_out_of_60_epochs_multi_loss_documentation_model.pth",
    repo_type="model",
    local_dir="/content"
)

print(f"Checkpoint downloaded to: {checkpoint_path}")

early_stopping_56_out_of_60_epochs_multi(…): reconstructing file:   0%|          |  0.00B /  297MB            

early_stopping_56_out_of_60_epochs_multi(…): downloading bytes:           |  0.00B            

Checkpoint downloaded to: /content/early_stopping_56_out_of_60_epochs_multi_loss_documentation_model.pth


In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device)

model = DepthAnythingV2(**{**model_configs[encoder], "max_depth": max_depth})
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device).eval()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"{type(model).__name__} loaded | {trainable:,} / {total:,} parameters trainable")

DepthAnythingV2 loaded | 24,785,089 / 24,785,089 parameters trainable


In [ ]:
#!/usr/bin/env python3
"""
Standalone offline depth evaluation for Depth Anything V2 — same metric
math and CSV format as offline_depth_eval.py's YOLO26 version (and the
ROS DepthCSVSummary node), so results are directly comparable across
both model families. Runs directly over (color_path, depth_path) pairs.

Usage (inside your Kaggle/Colab notebook, after training finishes):

    from offline_depth_eval_dav2 import run_offline_eval_dav2

    run_offline_eval_dav2(
        model_path="/kaggle/input/models/wasiqsaleem/dav2/pytorch/default/1/depth_anything_v2_metric_hypersim_vits.pth",
        test_samples=test_samples,          # list of (color_path, depth_path)
        output_csv="/kaggle/working/base_dav2_scannet_test_eval.csv",
        encoder="vits",
        depth_scale=1.0 / 1000.0,           # ScanNet raw depth is mm -> m
        min_depth=0.1,
        max_depth=20.0,
    )

Note: this expects the Depth-Anything-V2 repo (the `depth_anything_v2`
package) to be importable — same as in your fine-tuning notebooks, e.g.
sys.path.append('/kaggle/input/.../Depth-Anything-V2') or
sys.path.append('/kaggle/input/.../Depth-Anything-V2/metric_depth')
before importing this module.
"""

import os
import csv
import numpy as np
import torch
import cv2
from PIL import Image as PILImage


# Same architecture configs as Depth-Anything-V2's own scripts
DAV2_MODEL_CONFIGS = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
}


def load_dav2_model(model_path, encoder='vits', max_depth=20.0, device=None):
    """Load a Depth Anything V2 metric-depth model from a checkpoint.

    Handles both a bare state_dict (like the stock Hypersim release
    checkpoints) and a full training-checkpoint dict that also carries
    epoch/optimizer/loss keys (like your fine-tuned checkpoints) — adjust
    the key name below if your training loop saved it under something
    other than 'model_state_dict'.
    """
    from depth_anything_v2.dpt import DepthAnythingV2

    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

    model = DepthAnythingV2(**{**DAV2_MODEL_CONFIGS[encoder], 'max_depth': max_depth})

    checkpoint = torch.load(model_path, map_location='cpu')

    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint

    model.load_state_dict(state_dict)
    return model.to(device).eval()


def run_offline_eval_dav2(
    model_path,
    test_samples,
    output_csv,
    encoder='vits',
    depth_scale=1.0 / 1000.0,
    min_depth=0.1,
    max_depth=20.0,
    device=None,
):
    print(f"Loading model: {model_path}")
    model = load_dav2_model(model_path, encoder=encoder, max_depth=max_depth, device=device)

    os.makedirs(os.path.dirname(output_csv), exist_ok=True)

    fieldnames = [
        'frame_id', 'color_path', 'gt_mean', 'pred_mean', 'diff_mean',
        'AbsRel', 'RMSE', 'LogRMSE', 'Delta1', 'Delta2', 'Delta3',
        'valid_pixel_frac',
    ]

    rows = []
    n_skipped = 0

    for frame_id, (color_path, depth_path) in enumerate(test_samples):
        try:

            gt_raw = PILImage.open(depth_path)
            gt = np.array(gt_raw).astype(np.float32)

            # ScanNet raw depth is typically uint16 millimeters.
            if np.array(gt_raw).dtype == np.uint16:
                gt = gt * depth_scale
            raw_img = cv2.imread(color_path)
            if raw_img is None:
                print(f"[{frame_id}] Failed to read image: {color_path} — skipping.")
                n_skipped += 1
                continue

            with torch.no_grad():
                pred = model.infer_image(raw_img)

            pred = np.squeeze(np.asarray(pred))
            if pred.ndim != 2:
                print(f"[{frame_id}] Unexpected pred shape {pred.shape} — skipping.")
                n_skipped += 1
                continue
            if pred.shape != gt.shape:
                pred_img = PILImage.fromarray(pred)
                pred_img = pred_img.resize(
                    (gt.shape[1], gt.shape[0]), resample=PILImage.BILINEAR
                )
                pred = np.array(pred_img)

        except Exception as e:
            print(f"[{frame_id}] Failed to load/infer: {e}")
            n_skipped += 1
            continue
        mask = (gt >= min_depth) & (gt <= max_depth) & \
               (pred >= min_depth) & (pred <= max_depth)

        if np.count_nonzero(mask) == 0:
            print(f"[{frame_id}] No valid pixels after masking — skipping.")
            n_skipped += 1
            continue

        gt_valid = gt[mask]
        pred_valid = pred[mask]
        diff = np.abs(gt_valid - pred_valid)

        abs_rel = np.mean(diff / gt_valid)
        rmse = np.sqrt(np.mean((gt_valid - pred_valid) ** 2))

        gt_log = np.log(np.clip(gt_valid, 1e-6, None))
        pred_log = np.log(np.clip(pred_valid, 1e-6, None))
        log_rmse = np.sqrt(np.mean((gt_log - pred_log) ** 2))

        max_ratio = np.maximum(gt_valid / pred_valid, pred_valid / gt_valid)
        delta1 = np.mean(max_ratio < 1.25)
        delta2 = np.mean(max_ratio < 1.25 ** 2)
        delta3 = np.mean(max_ratio < 1.25 ** 3)

        row = {
            'frame_id': frame_id,
            'color_path': color_path,
            'gt_mean': float(np.mean(gt_valid)),
            'pred_mean': float(np.mean(pred_valid)),
            'diff_mean': float(np.mean(diff)),
            'AbsRel': float(abs_rel),
            'RMSE': float(rmse),
            'LogRMSE': float(log_rmse),
            'Delta1': float(delta1),
            'Delta2': float(delta2),
            'Delta3': float(delta3),
            'valid_pixel_frac': float(np.count_nonzero(mask) / mask.size),
        }
        rows.append(row)

        if frame_id % 50 == 0:
            print(f"Frame {frame_id}/{len(test_samples)} | "
                  f"AbsRel: {abs_rel:.4f} | RMSE: {rmse:.4f} | D1: {delta1:.4f}")

    # ---------------- Write CSV ----------------
    with open(output_csv, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nWrote {len(rows)} rows to {output_csv} ({n_skipped} skipped)")

    if rows:
        keys = ['AbsRel', 'RMSE', 'LogRMSE', 'Delta1', 'Delta2', 'Delta3']
        means = {k: np.mean([r[k] for r in rows]) for k in keys}
        print(f"\n=== Aggregate over {len(rows)} frames ===")
        for k, v in means.items():
            print(f"{k}: {v:.4f}")
        return means
    else:
        print("No frames evaluated — nothing to summarize.")
        return None

In [ ]:
means = run_offline_eval_dav2(
    model_path="/content/early_stopping_56_out_of_60_epochs_multi_loss_documentation_model.pth",
    test_samples=test_samples,
    output_csv="/content/base_dav2_scannet_test_eval.csv",
    encoder="vits",
    depth_scale=1.0 / 1000.0,
    min_depth=0.1,
    max_depth=20.0,
)

Loading model: /content/early_stopping_56_out_of_60_epochs_multi_loss_documentation_model.pth
Frame 0/902 | AbsRel: 0.1223 | RMSE: 0.2407 | D1: 0.9705
Frame 50/902 | AbsRel: 0.7349 | RMSE: 0.8218 | D1: 0.1019
Frame 100/902 | AbsRel: 0.2802 | RMSE: 0.4663 | D1: 0.4379
Frame 150/902 | AbsRel: 0.1681 | RMSE: 0.4245 | D1: 0.8351
Frame 200/902 | AbsRel: 0.1141 | RMSE: 0.2782 | D1: 0.8107
Frame 250/902 | AbsRel: 0.1881 | RMSE: 0.4795 | D1: 0.7933
Frame 300/902 | AbsRel: 0.0653 | RMSE: 0.2091 | D1: 0.9580
Frame 350/902 | AbsRel: 0.0920 | RMSE: 0.3111 | D1: 0.9697
Frame 400/902 | AbsRel: 0.1754 | RMSE: 0.5250 | D1: 0.6847
Frame 450/902 | AbsRel: 0.1276 | RMSE: 0.2478 | D1: 0.9727
Frame 500/902 | AbsRel: 0.1375 | RMSE: 0.3303 | D1: 0.9294
Frame 550/902 | AbsRel: 0.1175 | RMSE: 0.1533 | D1: 0.9652
Frame 600/902 | AbsRel: 0.1866 | RMSE: 0.3255 | D1: 0.6872
Frame 650/902 | AbsRel: 0.0733 | RMSE: 0.1316 | D1: 0.9839
Frame 700/902 | AbsRel: 0.1827 | RMSE: 0.9657 | D1: 0.5607
Frame 750/902 | AbsRel: 

## 4. OpenVINO IR Evaluation (Both Models)

Evaluates quantized `.xml`/`.bin` OpenVINO IR checkpoints directly through the OpenVINO runtime (`ov.Core`), bypassing both Ultralytics' and DAv2's own inference wrappers — so preprocessing has to be replicated manually here to match what each model actually saw during training/export. See the note in `run_openvino_depth_eval` below on why `imagenet_normalize` matters for DAv2 but not YOLO26.

In [ ]:
!pip install -q openvino

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 MB 14.0 MB/s eta 0:00:00


In [ ]:
import os
import csv
import cv2
import numpy as np
import openvino as ov
from PIL import Image as PILImage

In [ ]:
def run_openvino_depth_eval(model_xml, test_samples, output_csv, depth_scale=1.0 / 1000.0,
                             min_depth=0.1, max_depth=20.0, device="AUTO",
                             imagenet_normalize=False,
                             imagenet_mean=(0.485, 0.456, 0.406),
                             imagenet_std=(0.229, 0.224, 0.225)):
    print(f"Loading OpenVINO model: {model_xml}")

    core = ov.Core()
    model = core.read_model(model=model_xml)
    compiled_model = core.compile_model(model, device)

    input_layer = compiled_model.input(0)
    input_shape = list(input_layer.shape)

    print("Input shape:", input_shape)
    print("Output shape:", compiled_model.output(0).shape)

    rows = []
    n_skipped = 0

    fieldnames = [
        "frame_id", "color_path", "gt_mean", "pred_mean", "diff_mean",
        "AbsRel", "RMSE", "LogRMSE", "Delta1", "Delta2", "Delta3",
        "valid_pixel_frac",
    ]

    for frame_id, (color_path, depth_path) in enumerate(test_samples):

        try:
            gt_raw = PILImage.open(depth_path)
            gt_array = np.array(gt_raw)
            gt = gt_array.astype(np.float32)

            if gt_array.dtype == np.uint16:
                gt *= depth_scale

            image = cv2.imread(color_path)

            if image is None:
                print(f"[{frame_id}] Failed to read image — skipping.")
                n_skipped += 1
                continue
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            original_h, original_w = image.shape[:2]
            input_h = input_shape[-2]
            input_w = input_shape[-1]

            resized = cv2.resize(image, (input_w, input_h), interpolation=cv2.INTER_LINEAR)
            input_image = resized.astype(np.float32) / 255.0

            if imagenet_normalize:
                mean = np.array(imagenet_mean, dtype=np.float32)
                std = np.array(imagenet_std, dtype=np.float32)
                input_image = (input_image - mean) / std
            input_image = np.transpose(input_image, (2, 0, 1))
            input_image = np.expand_dims(input_image, axis=0)

            result = compiled_model([input_image])

            output = result[compiled_model.output(0)]
            pred = np.squeeze(output).astype(np.float32)

            if pred.ndim != 2:
                print(f"[{frame_id}] Unexpected prediction shape {pred.shape} — skipping.")
                n_skipped += 1
                continue

            if pred.shape != gt.shape:
                pred = cv2.resize(
                    pred,
                    (gt.shape[1], gt.shape[0]),
                    interpolation=cv2.INTER_LINEAR,
                )

        except Exception as e:
            print(f"[{frame_id}] Failed to load/infer: {e}")
            n_skipped += 1
            continue

        # Valid mask
        mask = (
            (gt >= min_depth)
            & (gt <= max_depth)
            & (pred >= min_depth)
            & (pred <= max_depth)
        )

        if np.count_nonzero(mask) == 0:
            print(f"[{frame_id}] No valid pixels — skipping.")
            n_skipped += 1
            continue

        gt_valid = gt[mask]
        pred_valid = pred[mask]

        diff = np.abs(gt_valid - pred_valid)

        abs_rel = np.mean(diff / gt_valid)
        rmse = np.sqrt(np.mean((gt_valid - pred_valid) ** 2))

        gt_log = np.log(np.clip(gt_valid, 1e-6, None))
        pred_log = np.log(np.clip(pred_valid, 1e-6, None))

        log_rmse = np.sqrt(np.mean((gt_log - pred_log) ** 2))

        max_ratio = np.maximum(
            gt_valid / pred_valid,
            pred_valid / gt_valid,
        )

        delta1 = np.mean(max_ratio < 1.25)
        delta2 = np.mean(max_ratio < 1.25 ** 2)
        delta3 = np.mean(max_ratio < 1.25 ** 3)

        rows.append({
            "frame_id": frame_id,
            "color_path": color_path,
            "gt_mean": float(np.mean(gt_valid)),
            "pred_mean": float(np.mean(pred_valid)),
            "diff_mean": float(np.mean(diff)),
            "AbsRel": float(abs_rel),
            "RMSE": float(rmse),
            "LogRMSE": float(log_rmse),
            "Delta1": float(delta1),
            "Delta2": float(delta2),
            "Delta3": float(delta3),
            "valid_pixel_frac": float(np.count_nonzero(mask) / mask.size),
        })

        if frame_id % 50 == 0:
            print(
                f"Frame {frame_id}/{len(test_samples)} | "
                f"AbsRel: {abs_rel:.4f} | "
                f"RMSE: {rmse:.4f} | "
                f"D1: {delta1:.4f}"
            )

    output_dir = os.path.dirname(output_csv)

    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    with open(output_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nWrote {len(rows)} rows to {output_csv} ({n_skipped} skipped)")

    if not rows:
        print("No frames evaluated.")
        return None

    keys = [
        "AbsRel",
        "RMSE",
        "LogRMSE",
        "Delta1",
        "Delta2",
        "Delta3",
    ]

    means = {
        k: float(np.mean([r[k] for r in rows]))
        for k in keys
    }

    print(f"\n=== Aggregate over {len(rows)} frames ===")

    for k, v in means.items():
        print(f"{k}: {v:.4f}")

    return means

### YOLO26-Depth (Small, Nano) — ScanNet-stage checkpoints

Ultralytics' export bakes in plain 0-255 → 0-1 scaling with no ImageNet mean/std subtraction, so `imagenet_normalize` is left at its default (`False`) for both calls below.

In [ ]:
from huggingface_hub import hf_hub_download

checkpoint_path = hf_hub_download(
    repo_id="WasiqSaleem/Fine-Tuned-Depth-Models-OpenVINO",
    filename="small_yolo26-depth-scannet-batch3.xml",
    repo_type="model",
    local_dir="/content"
)

print(f"Checkpoint downloaded to: {checkpoint_path}")

from huggingface_hub import hf_hub_download

bin_path = hf_hub_download(
    repo_id="WasiqSaleem/Fine-Tuned-Depth-Models-OpenVINO",
    filename="small_yolo26-depth-scannet-batch3.bin",
    repo_type="model",
    local_dir="/content"
)

print(f"Weights downloaded to: {bin_path}")

small_yolo26-depth-scannet-batch3.xml:   0%|          | 0.00/309k [00:00<?, ?B/s]

Checkpoint downloaded to: /content/small_yolo26-depth-scannet-batch3.xml


small_yolo26-depth-scannet-batch3.bin: reconstructing file:   0%|          |  0.00B / 48.1MB            

small_yolo26-depth-scannet-batch3.bin: downloading bytes:           |  0.00B            

Weights downloaded to: /content/small_yolo26-depth-scannet-batch3.bin


In [ ]:
yolo_openvino_means = run_openvino_depth_eval(
    model_xml="/content/small_yolo26-depth-scannet-batch3.xml",
    test_samples=test_samples,
    output_csv="/content/yolos_scannet_testing.csv",
    depth_scale=1.0 / 1000.0,
    min_depth=0.1,
    max_depth=20.0,
)

Loading OpenVINO model: /content/small_yolo26-depth-scannet-batch3.xml
Input shape: [1, 3, 640, 640]
Output shape: [1,1,640,640]
Frame 0/902 | AbsRel: 0.1166 | RMSE: 0.2309 | D1: 0.8767
Frame 50/902 | AbsRel: 0.0760 | RMSE: 0.1219 | D1: 0.9584
Frame 100/902 | AbsRel: 0.1030 | RMSE: 0.1954 | D1: 0.8995
Frame 150/902 | AbsRel: 0.2141 | RMSE: 0.6778 | D1: 0.4410
Frame 200/902 | AbsRel: 0.1482 | RMSE: 0.5890 | D1: 0.7360
Frame 250/902 | AbsRel: 0.0968 | RMSE: 0.3288 | D1: 0.9266
Frame 300/902 | AbsRel: 0.1502 | RMSE: 0.3685 | D1: 0.9402
Frame 350/902 | AbsRel: 0.0626 | RMSE: 0.1989 | D1: 0.9920
Frame 400/902 | AbsRel: 0.0771 | RMSE: 0.1641 | D1: 0.9643
Frame 450/902 | AbsRel: 0.0565 | RMSE: 0.1270 | D1: 0.9818
Frame 500/902 | AbsRel: 0.1294 | RMSE: 0.3160 | D1: 0.9292
Frame 550/902 | AbsRel: 0.0646 | RMSE: 0.0917 | D1: 0.9763
Frame 600/902 | AbsRel: 0.0715 | RMSE: 0.1277 | D1: 0.9991
Frame 650/902 | AbsRel: 0.2387 | RMSE: 0.4168 | D1: 0.4134
Frame 700/902 | AbsRel: 0.2543 | RMSE: 1.0390 | 

In [ ]:
from huggingface_hub import hf_hub_download

checkpoint_path = hf_hub_download(
    repo_id="WasiqSaleem/Fine-Tuned-Depth-Models-OpenVINO",
    filename="nano_yolo26-depth-scannet-batch3.xml",
    repo_type="model",
    local_dir="/content"
)

print(f"Checkpoint downloaded to: {checkpoint_path}")

from huggingface_hub import hf_hub_download

bin_path = hf_hub_download(
    repo_id="WasiqSaleem/Fine-Tuned-Depth-Models-OpenVINO",
    filename="nano_yolo26-depth-scannet-batch3.bin",
    repo_type="model",
    local_dir="/content"
)

print(f"Weights downloaded to: {bin_path}")

nano_yolo26-depth-scannet-batch3.xml:   0%|          | 0.00/308k [00:00<?, ?B/s]

Checkpoint downloaded to: /content/nano_yolo26-depth-scannet-batch3.xml


nano_yolo26-depth-scannet-batch3.bin: reconstructing file:   0%|          |  0.00B / 20.7MB            

nano_yolo26-depth-scannet-batch3.bin: downloading bytes:           |  0.00B            

Weights downloaded to: /content/nano_yolo26-depth-scannet-batch3.bin


In [ ]:
yolo_openvino_means = run_openvino_depth_eval(
    model_xml="/content/nano_yolo26-depth-scannet-batch3.xml",
    test_samples=test_samples,
    output_csv="/content/yolo_nano_scannet_testing.csv",
    depth_scale=1.0 / 1000.0,
    min_depth=0.1,
    max_depth=20.0,
)

Loading OpenVINO model: /content/nano_yolo26-depth-scannet-batch3.xml
Input shape: [1, 3, 640, 640]
Output shape: [1,1,640,640]
Frame 0/902 | AbsRel: 0.1117 | RMSE: 0.2351 | D1: 0.8441
Frame 50/902 | AbsRel: 0.1646 | RMSE: 0.2082 | D1: 0.7378
Frame 100/902 | AbsRel: 0.0932 | RMSE: 0.1705 | D1: 0.9777
Frame 150/902 | AbsRel: 0.2274 | RMSE: 0.5881 | D1: 0.5767
Frame 200/902 | AbsRel: 0.1323 | RMSE: 0.4845 | D1: 0.7743
Frame 250/902 | AbsRel: 0.1232 | RMSE: 0.4364 | D1: 0.8053
Frame 300/902 | AbsRel: 0.1608 | RMSE: 0.4036 | D1: 0.8681
Frame 350/902 | AbsRel: 0.0631 | RMSE: 0.1618 | D1: 0.9963
Frame 400/902 | AbsRel: 0.0544 | RMSE: 0.1402 | D1: 0.9784
Frame 450/902 | AbsRel: 0.1520 | RMSE: 0.3122 | D1: 0.8190
Frame 500/902 | AbsRel: 0.1715 | RMSE: 0.4127 | D1: 0.7296
Frame 550/902 | AbsRel: 0.0955 | RMSE: 0.1343 | D1: 0.8686
Frame 600/902 | AbsRel: 0.1050 | RMSE: 0.1756 | D1: 0.9746
Frame 650/902 | AbsRel: 0.1725 | RMSE: 0.3087 | D1: 0.6696
Frame 700/902 | AbsRel: 0.1737 | RMSE: 1.0681 | D

### Depth Anything V2 — OpenVINO checkpoints

**`imagenet_normalize=True`** is required for both calls below — DAv2's training/fine-tuning pipeline always applied `A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)` before the model saw an image, and the ONNX/OpenVINO export has no normalization layer built into the graph, so it still expects that same normalized input. Skipping this (as an earlier version of this function did) silently feeds the model out-of-distribution input and depresses every accuracy metric — before drawing any conclusion about a DAv2 checkpoint's real quality from a `run_openvino_depth_eval` result, confirm this flag was actually set.

In [ ]:
from huggingface_hub import hf_hub_download

checkpoint_path = hf_hub_download(
    repo_id="WasiqSaleem/Fine-Tuned-Depth-Models-OpenVINO",
    filename="quantized_early_stopping_56_out_of_60_epochs_multi_loss_documenatation_model.xml",
    repo_type="model",
    local_dir="/content"
)

print(f"Checkpoint downloaded to: {checkpoint_path}")

from huggingface_hub import hf_hub_download

bin_path = hf_hub_download(
    repo_id="WasiqSaleem/Fine-Tuned-Depth-Models-OpenVINO",
    filename="quantized_early_stopping_56_out_of_60_epochs_multi_loss_documenatation_model.bin",
    repo_type="model",
    local_dir="/content"
)

print(f"Weights downloaded to: {bin_path}")

(…)ochs_multi_loss_documenatation_model.xml:   0%|          | 0.00/892k [00:00<?, ?B/s]

Checkpoint downloaded to: /content/quantized_early_stopping_56_out_of_60_epochs_multi_loss_documenatation_model.xml


quantized_early_stopping_56_out_of_60_ep(…): reconstructing file:   0%|          |  0.00B / 25.6MB            

quantized_early_stopping_56_out_of_60_ep(…): downloading bytes:           |  0.00B            

Weights downloaded to: /content/quantized_early_stopping_56_out_of_60_epochs_multi_loss_documenatation_model.bin


In [ ]:
dav2_openvino_means = run_openvino_depth_eval(
    model_xml="/content/quantized_early_stopping_56_out_of_60_epochs_multi_loss_documenatation_model.xml",
    test_samples=test_samples,
    output_csv="/content/nyuv2_dav2_openvino_eval.csv",
    depth_scale=1.0 / 1000.0,
    min_depth=0.1,
    max_depth=20.0,
    imagenet_normalize=True,  # DAv2 was trained on ImageNet-normalized RGB input
)



Loading OpenVINO model: /content/quantized_early_stopping_56_out_of_60_epochs_multi_loss_documenatation_model.xml
Input shape: [1, 3, 364, 364]
Output shape: [1,364,364]
Frame 0/902 | AbsRel: 0.5974 | RMSE: 0.9540 | D1: 0.0212
Frame 50/902 | AbsRel: 1.4904 | RMSE: 1.5566 | D1: 0.0000
Frame 100/902 | AbsRel: 0.7666 | RMSE: 1.2121 | D1: 0.0011
Frame 150/902 | AbsRel: 0.3226 | RMSE: 0.6980 | D1: 0.6858
Frame 200/902 | AbsRel: 0.5976 | RMSE: 0.7977 | D1: 0.2368
Frame 250/902 | AbsRel: 0.4912 | RMSE: 0.9156 | D1: 0.0896
Frame 300/902 | AbsRel: 0.3921 | RMSE: 0.8555 | D1: 0.1089
Frame 350/902 | AbsRel: 0.4349 | RMSE: 0.9703 | D1: 0.0061
Frame 400/902 | AbsRel: 0.6114 | RMSE: 1.1863 | D1: 0.0000
Frame 450/902 | AbsRel: 0.7832 | RMSE: 1.0199 | D1: 0.0088
Frame 500/902 | AbsRel: 0.1836 | RMSE: 0.4360 | D1: 0.8520
Frame 550/902 | AbsRel: 0.9110 | RMSE: 1.0539 | D1: 0.0000
Frame 600/902 | AbsRel: 0.7204 | RMSE: 1.0207 | D1: 0.0001
Frame 650/902 | AbsRel: 0.5213 | RMSE: 0.7693 | D1: 0.0193
Frame 7

In [ ]:
from huggingface_hub import hf_hub_download

checkpoint_path = hf_hub_download(
    repo_id="WasiqSaleem/Fine-Tuned-Depth-Models-OpenVINO",
    filename="20_scannet_scannet_quantized_model.xml",
    repo_type="model",
    local_dir="/content"
)

print(f"Checkpoint downloaded to: {checkpoint_path}")

from huggingface_hub import hf_hub_download

bin_path = hf_hub_download(
    repo_id="WasiqSaleem/Fine-Tuned-Depth-Models-OpenVINO",
    filename="20_scannet_scannet_quantized_model.bin",
    repo_type="model",
    local_dir="/content"
)

print(f"Weights downloaded to: {bin_path}")

20_scannet_scannet_quantized_model.xml:   0%|          | 0.00/893k [00:00<?, ?B/s]

Checkpoint downloaded to: /content/20_scannet_scannet_quantized_model.xml


20_scannet_scannet_quantized_model.bin: reconstructing file:   0%|          |  0.00B / 26.7MB            

20_scannet_scannet_quantized_model.bin: downloading bytes:           |  0.00B            

Weights downloaded to: /content/20_scannet_scannet_quantized_model.bin


In [ ]:
dav2_openvino_means = run_openvino_depth_eval(
    model_xml="/content/20_scannet_scannet_quantized_model.xml",
    test_samples=test_samples,
    output_csv="/content/scannet_dav2_openvino_eval.csv",
    depth_scale=1.0 / 1000.0,
    min_depth=0.1,
    max_depth=20.0,
    imagenet_normalize=True,  # DAv2 was trained on ImageNet-normalized RGB input
)


Loading OpenVINO model: /content/20_scannet_scannet_quantized_model.xml
Input shape: [1, 3, 518, 518]
Output shape: [1,518,518]
Frame 0/902 | AbsRel: 0.2904 | RMSE: 0.4616 | D1: 0.4521
Frame 50/902 | AbsRel: 0.5427 | RMSE: 0.5599 | D1: 0.0808
Frame 100/902 | AbsRel: 0.2760 | RMSE: 0.4368 | D1: 0.3181
Frame 150/902 | AbsRel: 0.1821 | RMSE: 0.4066 | D1: 0.7734
Frame 200/902 | AbsRel: 0.3201 | RMSE: 0.4350 | D1: 0.3121
Frame 250/902 | AbsRel: 0.2863 | RMSE: 0.5306 | D1: 0.3233
Frame 300/902 | AbsRel: 0.1570 | RMSE: 0.3400 | D1: 0.9429
Frame 350/902 | AbsRel: 0.1866 | RMSE: 0.4120 | D1: 0.8659
Frame 400/902 | AbsRel: 0.2338 | RMSE: 0.4293 | D1: 0.5749
Frame 450/902 | AbsRel: 0.4078 | RMSE: 0.5200 | D1: 0.1896
Frame 500/902 | AbsRel: 0.1501 | RMSE: 0.3405 | D1: 0.9841
Frame 550/902 | AbsRel: 0.4281 | RMSE: 0.4858 | D1: 0.0270
Frame 600/902 | AbsRel: 0.3604 | RMSE: 0.5043 | D1: 0.0393
Frame 650/902 | AbsRel: 0.3153 | RMSE: 0.4688 | D1: 0.2145
Frame 700/902 | AbsRel: 0.2897 | RMSE: 0.5429 | D